# Phase 2 — Feature Engineering

**Goal of this notebook:** understand what we have, what needs fixing, and what we plan to build — before writing any transformation code.

---

## 1. Environment

Run this from the repo root with the project venv active (`.venv`).
Data lives in `titanic/data/raw/` — download it first if needed:

```bash
cd titanic && python setup_kaggle.py
```

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('../data/raw')

train = pd.read_csv(DATA / 'train.csv')
test  = pd.read_csv(DATA / 'test.csv')

print(f'train: {train.shape}   test: {test.shape}')

## 2. What we have

A quick look at types, nulls, and a sample row — nothing more.

In [ ]:
train.info()

In [ ]:
# Missing values in both splits — important to check test too, not just train
missing = pd.DataFrame({
    'train_missing': train.isnull().sum(),
    'test_missing':  test.isnull().sum(),
})
missing[missing.sum(axis=1) > 0]

## 3. Feature inventory

| Feature | Type | Missing | What to do |
|---|---|---|---|
| `Survived` | target (0/1) | — | keep as-is — **do not use as input to any imputation or feature** (target leakage) |
| `Pclass` | ordinal (1/2/3) | none | keep as-is |
| `Name` | string | none | **extract Title** (Mr, Mrs, Miss, Master, Rare) — strong survival signal and useful for age imputation |
| `Sex` | binary string | none | encode → 0/1 |
| `Age` | float | ~20% | **impute via multiple linear regression** using Pclass, Sex, Fare, Title — fit on rows with known age, predict missing; better than median because it borrows signal from all features simultaneously rather than relying on sparse manual subsets |
| `SibSp` | int | none | combine with Parch → **FamilySize** = SibSp + Parch + 1; also flag solo travellers |
| `Parch` | int | none | (see SibSp) |
| `Ticket` | string | none | mostly noise; can drop or extract ticket prefix — low priority |
| `Fare` | float | 1 in test | **bin into quartiles** (cheap/mid/expensive/luxury); correlated with Pclass so may add little |
| `Cabin` | string | ~77% | **extract deck letter** (A–G, T) where present; fill rest as 'U' (unknown) — too sparse to use in age imputation |
| `Embarked` | string | 2 in train | fill with mode ('S'); likely drop after — it's a proxy for class, not a causal feature |

**Features that will be dropped before modelling:** `PassengerId`, `Name` (after title extraction), `Ticket`, possibly `Cabin` if deck adds nothing.

**Order matters:** extract Title from Name *before* imputing Age — the regression needs Title as an input.

## 3a. Age imputation — why regression, not median

**The problem with median (even segmented median):** to use known relationships (older in 1st class, Masters are children, solo women in 1st class skew old) you'd need to slice by Pclass × Title × Sex × Fare simultaneously. Those buckets go sparse fast — a few passengers per cell — and a mean/median of 3 people is noise.

**What multiple linear regression buys you:** fit one model on all rows with known Age using `Pclass`, `Sex`, `Fare`, and extracted `Title` as inputs. The model learns a coefficient for each feature from *all* available data at once, then predicts the missing ages by plugging in each passenger's known values. Sparse buckets stop being a problem because the model borrows strength globally.

**The imputation pipeline (conceptual):**
```
1. Extract Title from Name  ← must happen first
2. Encode categorical inputs (Title, Sex) numerically
3. Fit LinearRegression on rows where Age is not null
4. Predict Age for rows where Age is null
5. Clip predictions to a sensible range (e.g. 0.5–80)
```

**Target leakage reminder:** `Survived` is not available in the test set at inference time — never use it as a feature, including inside imputation.

## 4. The pattern for other competitions

This notebook follows a repeatable structure that applies to any tabular classification competition:

1. **Load both splits together** — check missing values in *test* too, not just train. A feature missing in test at inference time is a silent bug.
2. **Inventory before transforming** — understand each column's type, cardinality, and null rate before writing any transformation.
3. **Impute with signal** — use known relationships (Title → Age) rather than global statistics where possible.
4. **Create, then evaluate** — build candidate features, then let the model (feature importance / permutation importance) tell you which ones actually help. Don't delete features by intuition alone.
5. **Apply identical transforms to test** — all imputation and encoding must fit on train only, then transform both. Fitting on the full dataset is data leakage.

The next notebook (`03_baseline.ipynb`) will take these engineered features and run a first model.